# TP2: Introduction to NLP with Python — Preprocessing
**Authors:** Eunice Agogo   
**Course:** Introduction to NLP (ECC Tutor — Asmaa Benzmane)

---

## Overview
This notebook covers **text normalization**, a crucial preprocessing stage in any NLP pipeline.  
We use **spaCy** (an industrial-strength NLP library) for tokenization and stop-word removal,  
alongside Python's built-in `re` module for pattern-based cleaning.

### Why normalize text?
Raw text is messy — it contains numbers, punctuation, varying capitalizations, and filler words  
that add no value to analysis. Normalization strips all of that away so a model sees only  
the meaningful vocabulary.


---
## Setup — Importing Libraries


In [ ]:
import re                                      # Python's built-in regular-expression library
import spacy                                   # Industrial NLP library — used for tokenization
from spacy.lang.en import English              # spaCy's lightweight English language model
from spacy.lang.en.stop_words import STOP_WORDS  # spaCy's built-in English stop-word list

# Create a blank English NLP pipeline.
# We use the blank model (no parser/NER) because we only need tokenization here.
nlp = English()

print("spaCy version :", spacy.__version__)
print("Number of English stop words in spaCy:", len(STOP_WORDS))
print("Sample stop words:", sorted(list(STOP_WORDS))[:15])


---
## Exercise 0 — Text Normalization Pipeline

We apply the following steps to a raw paragraph about Python's version history:

| Step | Operation | Tool used |
|------|-----------|-----------|
| 1 | Lowercase | Python `.lower()` |
| 2a | Remove numbers | `re.sub()` |
| 2b | Remove punctuation | `re.sub()` |
| 2c | Strip extra whitespace | `.strip()` |
| 2d | Tokenize (split into words) | **spaCy** |
| 2e | Remove stop words | spaCy `STOP_WORDS` |


In [ ]:
# ── Input text ─────────────────────────────────────────────────────────────
string = ("  Python 3.0, released in 2008, was a major revision of the "
          "language that is not completely backward compatible and much Python 2 "
          "code does not run unmodified on Python 3. With Python 2's end-of-life, "
          "only Python 3.6.x [30] and later are supported, with older versions "
          "still supporting e.g. Windows 7 (and old installers not restricted to "
          "64-bit Windows).")

print("ORIGINAL STRING:")
print(string)
print("\nLength (characters):", len(string))


### Step 1 — Convert to Lowercase
Every character is converted to its lowercase equivalent.  
This ensures "Python", "PYTHON", and "python" are treated as **the same word** by any downstream model.


In [ ]:
# string.lower() is a built-in Python string method.
# It iterates over every character and returns its lowercase version.
lower_string = string.lower()

print("AFTER LOWERCASING:")
print(lower_string)


### Step 2a — Remove Numbers
We use `re.sub(pattern, replacement, text)` from Python's `re` (regular expressions) module.

- **Pattern:** `r'\d+'`  
  - `\d` matches any digit character (0–9)  
  - `+` means "one or more of the preceding"  
  - So `\d+` matches any sequence of digits like `3`, `2008`, `3.6`, `64`  
- **Replacement:** `''` (empty string) — we delete the match  
- **Result:** All numbers are removed from the text


In [ ]:
# re.sub(pattern, replacement, string)
# r'\d+' is a raw string — the backslash is passed literally to the regex engine
# \d matches any digit (0-9); + means 'one or more digits in a row'
# We replace every digit sequence with '' (nothing), effectively deleting it
no_number_string = re.sub(r'\d+', '', lower_string)

print("AFTER REMOVING NUMBERS:")
print(no_number_string)


### Step 2b — Remove Punctuation
- **Pattern:** `r'[^\w\s]'`  
  - `[...]` is a character class — matches any one character inside the brackets  
  - `^` inside `[...]` means **NOT** — i.e. match anything that is NOT in the set  
  - `\w` matches word characters (letters, digits, underscore)  
  - `\s` matches whitespace characters (space, tab, newline)  
  - So `[^\w\s]` matches anything that is **neither a word character nor whitespace** → that's punctuation!  
- We replace every punctuation character with `''` (nothing)


In [ ]:
# [^\w\s] = NOT (word character OR whitespace) = punctuation
# This removes commas, periods, brackets, apostrophes, hyphens, etc.
no_punc_string = re.sub(r'[^\w\s]', '', no_number_string)

print("AFTER REMOVING PUNCTUATION:")
print(no_punc_string)


### Step 2c — Remove Extra Whitespace
After deleting numbers and punctuation, the text may have extra spaces  
(e.g. where "3.0" was, there is now "  ").  
`.strip()` removes leading and trailing whitespace.  
We could also use `re.sub(r'\s+', ' ', ...)` to collapse internal multiple spaces into one.


In [ ]:
# .strip() removes leading and trailing whitespace from the string
no_wspace_string = no_punc_string.strip()

# Also collapse any internal multiple spaces into a single space
no_wspace_string = re.sub(r'\s+', ' ', no_wspace_string)

print("AFTER STRIPPING WHITESPACE:")
print(no_wspace_string)


### Step 2d — Tokenize Using spaCy
**Tokenization** is the process of splitting a string into individual units (tokens).  
A token is usually a word, but can also be a punctuation mark or symbol.

We use **spaCy** here:
- `nlp(text)` processes the string and returns a `Doc` object
- Iterating over the `Doc` gives individual `Token` objects
- Each `Token` has a `.text` attribute (the raw string of that token)

spaCy's tokenizer is smarter than a simple `.split()` — it handles contractions,  
hyphenated words, and edge cases automatically.


In [ ]:
# nlp(no_wspace_string) runs the spaCy pipeline on our cleaned string
# This produces a 'Doc' object — a container of Token objects
doc = nlp(no_wspace_string)

# We build a list of the raw text of each token
lst_string = [token.text for token in doc]

print("TOKENIZED LIST (via spaCy):")
print(lst_string)
print("\nTotal tokens:", len(lst_string))


### Step 2e — Remove Stop Words
**Stop words** are very common words (like *is*, *the*, *and*, *not*, *with*)  
that carry little semantic meaning and would just add noise to analysis.

We use `spaCy`'s built-in `STOP_WORDS` set (326 words).  
A `set` is used instead of a list because checking membership (`in`) is O(1) for sets vs O(n) for lists.


In [ ]:
# We loop through every word in our token list
# If a word is NOT in spaCy's STOP_WORDS set AND is not just whitespace,
# we add it to our result string followed by a space
no_stpwords_string = ""
for i in lst_string:
    if i not in STOP_WORDS and i.strip() != '':   # keep only meaningful, non-empty tokens
        no_stpwords_string += i + ' '             # concatenate word + space

# After the loop, the string has one trailing space — remove it with .strip()
no_stpwords_string = no_stpwords_string.strip()

print("FINAL CLEANED STRING (stop words removed):")
print(no_stpwords_string)
print("\nWord count after full pipeline:", len(no_stpwords_string.split()))


### Pipeline Summary — Before vs After


In [ ]:
print("=" * 60)
print("BEFORE normalization:")
print(string.strip())
print()
print("AFTER normalization:")
print(no_stpwords_string)
print("=" * 60)
print(f"Original word count : {len(string.split())}")
print(f"Final word count    : {len(no_stpwords_string.split())}")
reduction = round((1 - len(no_stpwords_string.split()) / len(string.split())) * 100, 1)
print(f"Vocabulary reduced by {reduction}%")


---
## Question 1 — Why convert text to lowercase before analysis?

**Answer:**  
Natural language models and text algorithms are typically case-sensitive by default.  
Without lowercasing, `"Python"`, `"PYTHON"`, and `"python"` would be counted as **three distinct words**,  
even though they refer to the exact same thing.

This inflates vocabulary size unnecessarily and causes the model to miss that these tokens are related.  
By lowercasing first, we ensure **consistent representation** of the same word regardless of  
how the original author chose to capitalize it.

> Example in our text:  
> - `"Python"` appears 5 times with capital P  
> - `"PYTHON"` could also appear  
> - After lowercasing, all become `"python"` → counted and processed together


## Question 2 — Why remove numbers from text?

**Answer:**  
In most NLP tasks (sentiment analysis, topic modeling, text classification), numbers carry  
**no semantic value** — a model learning that `"2008"` or `"3.6"` appear in text gains nothing  
from those specific values.

Keeping numbers also increases vocabulary size with tokens that are unique and unrepeatable  
(every year, version number, or statistic is different), which makes generalization harder.

However, note that **in some tasks**, numbers *do* matter — e.g., financial NLP, date extraction,  
or named entity recognition. Removing them is context-dependent.

> In our text: `"3.0"`, `"2008"`, `"2"`, `"3.6"`, `"30"`, `"7"`, `"64"` — all version numbers  
> and years that don't help classify or analyze the *meaning* of the paragraph.


---
## Exercise 1 — Store Final Result as a List (not a string)

Instead of rebuilding a cleaned string, we now store the result directly as a **list of tokens**.  
This is actually more useful in NLP — most downstream tools (vectorizers, embeddings) expect lists.

I re-tokenize the cleaned (lowercase, no-numbers, no-punctuation) string using spaCy,  
then append only the meaningful tokens to `final_tokens`.


In [ ]:
# Re-process the cleaned string with spaCy to get Token objects
doc2 = nlp(no_wspace_string)

# Initialize an empty list to collect meaningful tokens
final_tokens = []

# Loop over every token in the spaCy Doc
for token in doc2:
    # token.text is the string representation of the token
    # We keep a token only if:
    #   1. it is NOT a stop word (not in spaCy's STOP_WORDS set)
    #   2. it is not blank/whitespace (token.text.strip() != '')
    if token.text not in STOP_WORDS and token.text.strip() != '':
        final_tokens.append(token.text)   # .append() adds the token to the end of the list

print("FINAL TOKENS AS A LIST:")
print(final_tokens)
print(f"\nTotal meaningful tokens: {len(final_tokens)}")


---
## Comparison: String output (Ex 0) vs List output (Ex 1)


In [ ]:
print("Exercise 0 output (string):")
print(type(no_stpwords_string))
print(no_stpwords_string)

print()
print("Exercise 1 output (list):")
print(type(final_tokens))
print(final_tokens)

print()
print("Why is a list more useful?")
print("  - Lists allow indexing:  final_tokens[0] =", final_tokens[0])
print("  - Lists allow slicing:   final_tokens[:5] =", final_tokens[:5])
print("  - Lists are the expected input for TF-IDF, Word2Vec, etc.")


---
## Bonus — Exploring spaCy's Token Attributes

spaCy `Token` objects expose many useful linguistic properties beyond just `.text`.  
Here we show a few that are relevant to preprocessing.


In [ ]:
# Process the original (non-cleaned) lowercased string with spaCy
# to show token attributes before any filtering
doc3 = nlp(lower_string[:200])   # use first 200 chars to keep output short

print(f"{'TOKEN':<20} {'IS_STOP':<12} {'IS_PUNCT':<12} {'IS_DIGIT':<12} {'IS_SPACE'}")
print("-" * 65)
for token in doc3:
    print(f"{token.text:<20} {str(token.is_stop):<12} {str(token.is_punct):<12} "
          f"{str(token.is_digit):<12} {str(token.is_space)}")


In [ ]:
# We can use these spaCy attributes directly instead of regex for a cleaner pipeline
# This is the 'spaCy way' to do what we did above:

doc4 = nlp(lower_string)

spacy_clean_tokens = [
    token.text
    for token in doc4
    if not token.is_stop       # remove stop words
    and not token.is_punct     # remove punctuation tokens
    and not token.is_digit     # remove digit-only tokens
    and not token.is_space     # remove whitespace tokens
]

print("spaCy-native cleaned tokens:")
print(spacy_clean_tokens)
print(f"\nCount: {len(spacy_clean_tokens)}")


---
## Summary

| Step | Method | Purpose |
|------|--------|---------|
| Lowercase | `str.lower()` | Unify case variants of the same word |
| Remove numbers | `re.sub(r'\d+', '', ...)` | Remove meaningless numeric tokens |
| Remove punctuation | `re.sub(r'[^\w\s]', '', ...)` | Remove non-alphabetic symbols |
| Strip whitespace | `str.strip()` + `re.sub(r'\s+', ' ', ...)` | Clean up gaps left by removal |
| Tokenize | `spaCy nlp(text)` → `[token.text for token in doc]` | Split into individual word units |
| Remove stop words | `token.text not in STOP_WORDS` | Remove common low-value words |

spaCy makes tokenization and stop-word removal both **fast and linguistically accurate**,  
making it the industry-standard choice for production NLP pipelines.
